In [ ]:
# Install (Run once)
!pip install -q tokenizers

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

# Import with aliases
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPreTokenizer
from tokenizers.processors import ByteLevel as ByteLevelProcessor

# ----------------------------------------------------------
# Train a tiny ByteLevel BPE tokenizer
# ----------------------------------------------------------
corpus = [
    "My name is John.",
    "John likes AI.",
    "Hello world."
]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevelPreTokenizer()

trainer = BpeTrainer(
    vocab_size=100,
    special_tokens=["[UNK]"]
)

tokenizer.train_from_iterator(corpus, trainer)

# ----------------------------------------------------------
# Input text
# ----------------------------------------------------------
text = "My name is John."

print("=" * 70)
print("Original Text:", repr(text))
print("=" * 70)

# ----------------------------------------------------------
# WITHOUT trim_offsets
# ----------------------------------------------------------
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=False)

encoding = tokenizer.encode(text)

print("\nWITHOUT trim_offsets=True")
print("-" * 70)

for token, (start, end) in zip(encoding.tokens, encoding.offsets):
    print(f"{token:10} Offset: ({start:2}, {end:2})   Span -> {repr(text[start:end])}")

# ----------------------------------------------------------
# WITH trim_offsets
# ----------------------------------------------------------
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=True)

encoding = tokenizer.encode(text)

print("\nWITH trim_offsets=True")
print("-" * 70)

for token, (start, end) in zip(encoding.tokens, encoding.offsets):
    print(f"{token:10} Offset: ({start:2}, {end:2})   Span -> {repr(text[start:end])}")

Original Text: 'My name is John.'

WITHOUT trim_offsets=True
----------------------------------------------------------------------
ĠMy        Offset: ( 0,  2)   Span -> 'My'
Ġname      Offset: ( 2,  7)   Span -> ' name'
Ġis        Offset: ( 7, 10)   Span -> ' is'
ĠJohn      Offset: (10, 15)   Span -> ' John'
.          Offset: (15, 16)   Span -> '.'

WITH trim_offsets=True
----------------------------------------------------------------------
ĠMy        Offset: ( 0,  2)   Span -> 'My'
Ġname      Offset: ( 3,  7)   Span -> 'name'
Ġis        Offset: ( 8, 10)   Span -> 'is'
ĠJohn      Offset: (11, 15)   Span -> 'John'
.          Offset: (15, 16)   Span -> '.'


In [ ]:
# WITHOUT trimming
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=False)
enc = tokenizer.encode(text)

for token, (start, end) in zip(enc.tokens, enc.offsets):
    if "John" in token:
        print("WITHOUT trim_offsets")
        print("Token :", token)
        print("Offset:", (start, end))
        print("Extracted Span:", repr(text[start:end]))

# WITH trimming
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=True)
enc = tokenizer.encode(text)

for token, (start, end) in zip(enc.tokens, enc.offsets):
    if "John" in token:
        print("\nWITH trim_offsets")
        print("Token :", token)
        print("Offset:", (start, end))
        print("Extracted Span:", repr(text[start:end]))

WITHOUT trim_offsets
Token : ĠJohn
Offset: (10, 15)
Extracted Span: ' John'

WITH trim_offsets
Token : ĠJohn
Offset: (11, 15)
Extracted Span: 'John'


In [ ]:
# Install (Run once)
#!pip install -q tokenizers

from tokenizers import Tokenizer, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

# -------------------------------------------------------
# Training corpus (contains only English)
# -------------------------------------------------------
corpus = [
    "Hello world",
    "Machine Learning",
    "Artificial Intelligence"
]

test_text = "Hello 😊 中"

# -------------------------------------------------------
# Tokenizer WITHOUT initial_alphabet
# -------------------------------------------------------
tok_without = Tokenizer(BPE(unk_token="[UNK]"))
tok_without.pre_tokenizer = pre_tokenizers.ByteLevel()

trainer_without = BpeTrainer(
    vocab_size=100,
    special_tokens=["[UNK]"]
)

tok_without.train_from_iterator(corpus, trainer_without)

# -------------------------------------------------------
# Tokenizer WITH initial_alphabet
# -------------------------------------------------------
tok_with = Tokenizer(BPE(unk_token="[UNK]"))
tok_with.pre_tokenizer = pre_tokenizers.ByteLevel()

trainer_with = BpeTrainer(
    vocab_size=100,
    special_tokens=["[UNK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

tok_with.train_from_iterator(corpus, trainer_with)

# -------------------------------------------------------
# Encode the same sentence
# -------------------------------------------------------
enc_without = tok_without.encode(test_text)
enc_with = tok_with.encode(test_text)

print("="*70)
print("Input Text:", repr(test_text))
print("="*70)

print("\nWITHOUT initial_alphabet")
print("-------------------------")
print(enc_without.tokens)

print("\nWITH initial_alphabet")
print("----------------------")
print(enc_with.tokens)

Input Text: 'Hello 😊 中'

WITHOUT initial_alphabet
-------------------------
['ĠHello', 'Ġ', '[UNK]', '[UNK]', '[UNK]', '[UNK]', 'Ġ', '[UNK]', '[UNK]', '[UNK]']

WITH initial_alphabet
----------------------
['Ġ', 'H', 'e', 'l', 'l', 'o', 'Ġ', 'ð', 'Ł', 'ĺ', 'Ĭ', 'Ġ', 'ä', '¸', 'Ń']


In [ ]:
print("WITHOUT initial_alphabet")
print(enc_without.tokens)
print("Contains [UNK] ?", "[UNK]" in enc_without.tokens)

print("\nWITH initial_alphabet")
print(enc_with.tokens)
print("Contains [UNK] ?", "[UNK]" in enc_with.tokens)

WITHOUT initial_alphabet
['ĠHello', 'Ġ', '[UNK]', '[UNK]', '[UNK]', '[UNK]', 'Ġ', '[UNK]', '[UNK]', '[UNK]']
Contains [UNK] ? True

WITH initial_alphabet
['Ġ', 'H', 'e', 'l', 'l', 'o', 'Ġ', 'ð', 'Ł', 'ĺ', 'Ĭ', 'Ġ', 'ä', '¸', 'Ń']
Contains [UNK] ? False
